# Bacpipe Tutorial
This notebook walks through the main workflows of the `bacpipe` library — from inspecting configuration and loading audio, through generating embeddings with built-in and custom models, to running the full pipeline and benchmarking classifier performance.

---
## 1. Setup & Configuration
Import necessary modules 

In [ ]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display
import os 
from pathlib import Path

# load the specific package
import bacpipe

Set the working directory to the repository root and clean the previous tests if needed/wanted

In [ ]:
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# replace this with the path to the directory on your machine
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# os.chdir('../..')
os.chdir('/media/haupert/data/mes_projets/_packages/bacpipe.git/bacpipe')

# Change the value of the key main_results_dir in the namespace bacpipe.settings to change the directory 
# where the results of the tutorials are stored. By default, it is set to './bacpipe_results'.
bacpipe.settings.main_results_dir = str(Path(bacpipe.settings.main_results_dir) / 'clustering')

# !WARNING! the following code deletes the folder where the results of this tutorial is stored to be sure to start with a clean folder. 
# If you have important data in this folder, please comment it before running this code.
folder_path = bacpipe.settings.main_results_dir
if os.path.exists(folder_path):
    # Prompt the user
    user_input = input(f"Are you sure you want to delete '{folder_path}'? (y/n): ").lower().strip()

    if user_input == 'y':
        import shutil
        shutil.rmtree(folder_path) 
        print(f"Folder {folder_path} deleted.")
    else:
        print("Operation cancelled.")

else:
    print(f"Folder {folder_path} not found.")

Set global constants that will use through the notebook.

In [ ]:
MODEL_NAME = 'birdnet'                  # name of the model to run. Supported models are in bacpipe.supported_models
AUDIO_DIR = 'bacpipe/tests/test_data'   # path to directory containing audio files

---
## 2. Clustering Pipeline
First compute the embeddings before performing the clustering pipeline


In [ ]:
embeds = bacpipe.run_pipeline_for_single_model(
    model_name=MODEL_NAME,              # name of the model to run. Supported models are in bacpipe.supported_models
    audio_dir=AUDIO_DIR,                # path to directory containing audio files  
).embeddings(return_type='array')

Run bacpipe's built-in clustering evaluation pipeline to group embeddings and compare the clusters against your ground truth labels.

In [ ]:
# !BUG We need to explaing the output of clustering_pipeline. 
# What are all the metrics, especially the prefix difference between kmeans- and species-
# Is kmeans- metrics computed with the clustering of the kmeans algorithm 
# and species- metrics computed with the clustering aligned with the ground truth labels?

# Run this function after computing the embeddings 
# otherwise it is no able to find the connection between embeddings and ground truth labels
gt = bacpipe.ground_truth_by_model(
    model=MODEL_NAME, 
    audio_dir=AUDIO_DIR, 
    annotations_filename='annotations.csv',     # By default, it looks for the annotations file in AUDIO_DIR
    single_label=False,
    overwrite=False
)

# Run the clustering pipeline with the embeddings and the ground truth labels
clusterings, metrics = bacpipe.clustering_pipeline(
    model_name=MODEL_NAME, 
    ground_truth=gt,
    embeds=embeds
)

# display the metrics of the clustering
display(metrics)

---
## 3. Custom Clustering & Evaluation
Inject your own custom clustering algorithms (like sklearn's KMeans) and evaluate their performance using metrics like the Silhouette score.

In [ ]:
# ! BUG 
# it is not clear on what the silhouette score is computed. I change the clustering config but the silhouette score does not change. 
# The array for the ground truth was not correct, I changed the code

from sklearn.cluster import KMeans

# Make sure embeddings are generated
loader_obj = bacpipe.generate_embeddings(
    model_name=MODEL_NAME,
    audio_dir=AUDIO_DIR
)

# Get the embeddings as a numpy array
embeds = loader_obj.embeddings(return_type='array')

# Run custom clustering without ground truth
clustered_points = bacpipe.run_clustering(
    embeds=embeds, 
    cluster_configs={'my_clustering': KMeans(n_clusters=6) } 
)

# Run custom clustering with ground truth for alignment/evaluation
clustered_points_gt = bacpipe.run_clustering(
    embeds=embeds, 
    cluster_configs={'my_clustering': KMeans(n_clusters=6)}, 
    ground_truth=gt['label:species'][gt['label:species'] != -1]
)

# Evaluate embedding separation using Silhouette score
metrics = bacpipe.eval_with_silhouette(
    embeds=embeds, 
    # ground_truth=gt['label:species']
    ground_truth=gt['label:species'][gt['label:species'] != -1]
)

# display the silhouette score
display(metrics)

---
## 4. Display the clusters

In [ ]:
# !BUG Is there a better way to use the gt dict to get the labels for the points?

%matplotlib inline

# Use PCA to visualize the clusters in 2D
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# compute the PCA of the embeddings
pca = PCA(n_components=2)
embeds_2d = pca.fit_transform(embeds)

# Plot the PCA of the embeddings colored by the ground truth species labels
labels_num = gt['label:species'][gt['label:species'] != -1]
inverse_label_dict = {v: k for k, v in gt["label_dict:species"].items()} # Create inverse mapping from numeric labels to species names
labels_str = [inverse_label_dict[label] for label in labels_num]
unique_labels = np.unique(labels_num)

# set the figure size
plt.figure(figsize=(4, 3))

# Plot each species separately so they get their own label
for label_val in unique_labels:
    idx = np.where(labels_num == label_val)
    plt.scatter(
        embeds_2d[idx, 0], 
        embeds_2d[idx, 1], 
        label=inverse_label_dict[label_val],
        alpha=0.7 # Optional: transparency helps with overlap
    )

plt.title('PCA of Embeddings Colored by Species')
plt.legend(title="Species", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.show()


In [ ]:
# Use the clustered_points as label of the points in the PCA plot
# Plot the PCA of the embeddings colored by the clustered points
labels_num = clustered_points['my_clustering']
unique_labels = np.unique(labels_num)

# set the figure size
plt.figure(figsize=(4, 3))
for label_val in unique_labels:
    idx = np.where(labels_num == label_val)
    plt.scatter(
        embeds_2d[idx, 0], 
        embeds_2d[idx, 1], 
        label=f'Cluster {label_val}',
        alpha=0.7 # Optional: transparency helps with overlap
    )
plt.title('PCA of Embeddings Colored by Cluster number')
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.show()
